## Hawker Centre Inspection: Extension 1 (Time-Dependent Congestion)

The base model uses OSRM free-flow travel times. In reality, daytime traffic is slower, and how much slower depends on the time of day. Because each inspection takes a fixed time, the order of visits decides **when** each leg is driven, and therefore which congestion factor applies to it. This makes the problem a **time-dependent TSP**.

**Assumptions**

- The team leaves the depot (node 1) at 8:30 on a weekday and must inspect all 7 centres before returning.
- Each inspection takes 60 minutes, the same as in Extension 2. There is no lunch break: the team eats on the way.
- No waiting: the team starts inspecting as soon as it arrives and leaves as soon as it finishes.
- A leg's travel time is the free-flow time $t_{ij}$ multiplied by the congestion factor of the time band in which the leg **starts**. The whole leg uses that factor, even if it ends in the next band.

**Congestion factors**

The factors come from the TomTom Traffic Index 2025 for Singapore, which reports the average time to drive 10 km on each weekday in 2-hour bands. For each band, we average Monday to Friday and divide by the fastest band (06:00-08:00, about 14.8 min), which we use as a proxy for free-flow conditions. The same factor applies to every leg in a band.

This is a city-wide average, not data for our area, so the factors are an assumption informed by observed data.

**Sensitivity analysis**

We vary one input at a time, holding the others at their main values, and check whether the optimal route and total travel time change:

1. **Directional congestion**: commuter traffic is heavier in one direction (LTA charges ERP on the AYE towards the city in the morning). Legs from the west cluster $\{1,\dots,5\}$ to the east cluster $\{6,7,8\}$, or the reverse, get an extra factor $\beta \ge 1$ in one time band.
2. **Start time**: 7:00 to 10:00.
3. **Inspection time per centre**: 30 to 90 minutes.

### Model

**Sets and parameters**

- Nodes $i,j \in \{1,\dots,n\}$; node $1$ is the depot and nodes $2,\dots,n$ are the hawker centres ($n=8$).
- Time bands $k=1,\dots,K$, with band $k$ covering $[b_{k-1}, b_k)$, measured in minutes after midnight.
- $t_{ij}$: free-flow travel time from $i$ to $j$.
- $\alpha_{ijk}$: congestion factor for leg $i\to j$ when it starts in band $k$ (here the same for every leg: $\alpha_{ijk}=\alpha_k$).
- $d_i$: inspection time at node $i$ ($d_1=0$).
- $S$: start time at the depot (8:30, i.e. 510).
- $M$: a large constant (24 h = 1440 min).

**Decision variables**

- $x_{ij}\in\{0,1\}$: 1 if the team travels directly from $i$ to $j$.
- $p_{ik}\in\{0,1\}$: 1 if the team leaves node $i$ during band $k$.
- $s_i \ge 0$: time the team arrives at node $i$ and starts inspecting ($s_1 = S$).
- $T_j \ge 0$: travel time of the leg that enters node $j$ ($T_1$ is the return leg to the depot).

Let $q_i = s_i + d_i$ be the time the team leaves node $i$. Then $\tau_{ij}=\sum_k \alpha_{ijk}\,t_{ij}\,p_{ik}$ is the travel time of leg $i\to j$ given when the team leaves $i$.

**Objective: minimise total travel time**

$$\min \sum_{j=1}^{n} T_j$$

**Constraints**

Each node is left once and entered once:

$$\sum_{j\ne i} x_{ij}=1 \quad\forall i, \qquad \sum_{i\ne j} x_{ij}=1 \quad\forall j$$

Each departure falls in exactly one time band:

$$\sum_k p_{ik}=1, \qquad b_{k-1}-M(1-p_{ik}) \;\le\; q_i \;\le\; b_k-\varepsilon+M(1-p_{ik}) \qquad \forall i,k$$

Here $\varepsilon$ is a tiny constant (0.01 min, under one second). Without it, a departure exactly on a boundary (e.g. leaving at 10:00) would satisfy both neighbouring bands, and the model would pick the cheaper one. With it, band $k$ means $b_{k-1} \le q_i < b_k$, so a departure at 10:00 falls in the 10:00-12:00 band.

Time passes along the route with no waiting: if leg $i\to j$ is used, the team arrives at $j$ exactly $\tau_{ij}$ after leaving $i$. This is written as two big-M inequalities:

$$q_i+\tau_{ij}-M(1-x_{ij}) \;\le\; s_j \;\le\; q_i+\tau_{ij}+M(1-x_{ij}) \qquad \forall i\ne j,\; j\ge 2$$

Travel time of the leg entering $j$:

$$T_j \ge \tau_{ij}-M(1-x_{ij}) \qquad \forall i\ne j$$

Start time and domains:

$$s_1=S, \qquad x_{ij},\,p_{ik}\in\{0,1\}, \qquad s_i,\,T_j \ge 0$$

**Node labels:** the depot is node 1 and the hawker centres are nodes 2 to 8. Python arrays start at 0, so in the code the depot is index 0 and node $k$ is index $k-1$. Routes are printed and saved with the node labels (index + 1).

**Why no MTZ constraints are needed:** along any route, $s_j \ge s_i + d_i > s_i$, so arrival times strictly increase. A subtour among the centres would need a node to be visited after itself, which is impossible. The time variables play the role of the MTZ ordering variables.

**Why only the departure time matters:** with fixed inspection times and no waiting, the return time equals $S + \sum_j T_j + \sum_i d_i$. Minimising total travel time therefore also gives the earliest return.

### Imports

In [ ]:
from gurobipy import *
import itertools
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

### Load Data

In [ ]:
# Find the repo root (the folder that contains data/processed), whether the notebook
# runs from src/ or from the repo root
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here)]
             if os.path.isdir(os.path.join(p, "data", "processed"))), None)
if ROOT is None:
    raise FileNotFoundError(f"Cannot find data/processed from {here}. Put this notebook in the repo's src/ folder.")
print(f"Repo root: {ROOT}")

DATA_PATH = os.path.join(ROOT, "data", "processed")
EXTERNAL_PATH = os.path.join(ROOT, "data", "external")
RESULTS_PATH = os.path.join(ROOT, "results")
os.makedirs(RESULTS_PATH, exist_ok=True)

nodes_df = pd.read_csv(os.path.join(DATA_PATH, "nodes.csv"))
travel_times_df = pd.read_csv(os.path.join(DATA_PATH, "time_matrix_min.csv"), index_col=0)

travel_time_matrix = travel_times_df.to_numpy()
num_nodes = len(nodes_df)
short_names = nodes_df["short_name"].tolist()
print(f"Number of nodes (including depot): {num_nodes}")

### Setting Up Known Constants

In [ ]:
START_TIME = 8 * 60 + 30            # 8:30, in minutes after midnight
INSPECTION_TIME_PER_CENTRE = 60      # minutes, same as Extension 2

def make_inspection_times(minutes_per_centre):
    times = np.full(num_nodes, minutes_per_centre)
    times[0] = 0                     # no inspection at the depot (node 1, index 0)
    return times

inspection_times = make_inspection_times(INSPECTION_TIME_PER_CENTRE)
print(f"Inspection times (min): {inspection_times}")

# West cluster (Jurong / Boon Lay, incl. depot) = nodes 1-5; east cluster (Teban / West Coast / Clementi) = nodes 6-8.
# These are 0-based Python indices (node label = index + 1).
WEST = [0, 1, 2, 3, 4]
EAST = [5, 6, 7]
CITY_BOUND_ARCS = [(i, j) for i in WEST for j in EAST]   # towards the city
WEST_BOUND_ARCS = [(j, i) for i in WEST for j in EAST]   # away from the city

### Congestion Factors from TomTom

`tomtom_singapore_weekday_2025.csv` holds TomTom Traffic Index 2025 values for Singapore: average minutes to drive 10 km, by weekday and 2-hour band. Each band starts at `band_start` and lasts 2 hours. Values were read from the TomTom website (https://www.tomtom.com/traffic-index/singapore-traffic/) and rounded to the nearest 10 seconds.

In [ ]:
tomtom_df = pd.read_csv(os.path.join(EXTERNAL_PATH, "tomtom_singapore_weekday_2025.csv"))
weekdays = ["Mon", "Tue", "Wed", "Thu", "Fri"]
tomtom_df["weekday_avg_min"] = tomtom_df[weekdays].mean(axis=1)

# The fastest band (06:00-08:00) is used as the free-flow reference
free_flow_min = tomtom_df["weekday_avg_min"].min()
tomtom_df["factor"] = tomtom_df["weekday_avg_min"] / free_flow_min
tomtom_df["band_start_min"] = [int(h) * 60 + int(m) for h, m in tomtom_df["band_start"].str.split(":")]
print(f"Free-flow reference: {free_flow_min:.2f} min per 10 km")

# Keep the bands from 06:00 onwards (the earliest start time in the sensitivity analysis is 7:00)
day_df = tomtom_df[tomtom_df["band_start_min"] >= 6 * 60].reset_index(drop=True)
band_bounds = [int(b) for b in day_df["band_start_min"]] + [24 * 60]
band_factors = day_df["factor"].to_numpy()
band_labels = [f"{s // 60:02d}:00-{e // 60:02d}:00" for s, e in zip(band_bounds[:-1], band_bounds[1:])]

day_df[["band_start", "weekday_avg_min", "factor"]].round(3)

### Functions

In [ ]:
def build_alpha(band_factors, num_nodes, beta=1.0, beta_bands=(), beta_arcs=()):
    """alpha[i, j, k]: congestion factor for leg i -> j starting in band k.
    Every leg gets the band factor; legs in beta_arcs get an extra factor beta in beta_bands."""
    alpha = np.tile(band_factors, (num_nodes, num_nodes, 1)).astype(float)
    for k in beta_bands:
        for i, j in beta_arcs:
            alpha[i, j, k] *= beta
    return alpha


def find_band(t, band_bounds):
    for k in range(len(band_bounds) - 1):
        if band_bounds[k] <= t < band_bounds[k + 1]:
            return k
    raise ValueError(f"Time {t} is outside the modelled bands")


def evaluate_route(route, travel_times, inspection_times, alpha, band_bounds, start_time):
    """Drive a given route under time-dependent travel times.
    Returns total travel time, return time, and a leg-by-leg schedule."""
    t = start_time
    total_travel = 0.0
    legs = []
    for i, j in zip(route[:-1], route[1:]):
        k = find_band(t, band_bounds)
        leg_time = alpha[i, j, k] * travel_times[i, j]
        legs.append({"from": i, "to": j, "depart": t, "band": k,
                     "factor": alpha[i, j, k], "travel_min": leg_time})
        total_travel += leg_time
        t += leg_time + inspection_times[j]
    return total_travel, t, legs


def brute_force_td_tsp(travel_times, inspection_times, alpha, band_bounds, start_time):
    """Evaluate all (n-1)! routes (0-based indices). Used to check the MIP on our small instance."""
    n = len(travel_times)
    results = []
    for perm in itertools.permutations(range(1, n)):
        route = [0] + list(perm) + [0]
        travel, ret, _ = evaluate_route(route, travel_times, inspection_times, alpha, band_bounds, start_time)
        results.append((travel, ret, route))
    results.sort(key=lambda r: r[0])
    return results


def to_labels(route):
    """Convert 0-based indices to node labels (depot = node 1, centres = nodes 2-8)."""
    return [int(i) + 1 for i in route]


def to_clock(t):
    t = int(round(t))
    return f"{t // 60:02d}:{t % 60:02d}"

In [ ]:
def run_td_tsp(travel_times, inspection_times, alpha, band_bounds, start_time, M=24 * 60, eps=0.01, verbose=True):
    """Time-dependent TSP: minimise total travel time when each leg's travel time
    depends on the band in which it starts. Returns (route, total_travel_time)."""
    n = len(travel_times)
    K = alpha.shape[2]
    # Plain Python numbers, so numpy scalars never sit on the left of a Gurobi expression
    bounds = [float(b) for b in band_bounds]
    d = [float(v) for v in inspection_times]
    leg_cost = {(i, j, k): float(alpha[i, j, k] * travel_times[i, j])
                for i in range(n) for j in range(n) for k in range(K) if i != j}

    tsp = Model("Time-Dependent Hawker Centre Inspection")
    tsp.Params.OutputFlag = 1 if verbose else 0
    tsp.Params.MIPGap = 0          # prove optimality: the best routes differ by only a few hundredths of a minute
    tsp.Params.IntFeasTol = 1e-9   # stop big-M constraints from leaking time through near-integer binaries

    # Whether we travel directly from node i to node j
    x = tsp.addVars(n, n, vtype=GRB.BINARY, name="x")
    # Whether we leave node i during band k
    p = tsp.addVars(n, K, vtype=GRB.BINARY, name="p")
    # Arrival (= inspection start) time at node i
    s = tsp.addVars(n, lb=0, ub=M, name="s")
    # Travel time of the leg entering node j (T[0] is the return leg to the depot, node 1)
    T = tsp.addVars(n, lb=0, ub=M, name="T")

    # Departure time from node i, and travel time of leg i -> j given the departure band
    q = {i: s[i] + d[i] for i in range(n)}
    tau = {(i, j): quicksum(leg_cost[i, j, k] * p[i, k] for k in range(K))
           for i in range(n) for j in range(n) if i != j}

    tsp.setObjective(quicksum(T[j] for j in range(n)), GRB.MINIMIZE)

    # Visit constraints
    tsp.addConstrs((x[i, i] == 0 for i in range(n)), "no_self_loop")
    tsp.addConstrs((quicksum(x[i, j] for j in range(n)) == 1 for i in range(n)), "leave_once")
    tsp.addConstrs((quicksum(x[i, j] for i in range(n)) == 1 for j in range(n)), "enter_once")

    # Start time at the depot
    tsp.addConstr(s[0] == float(start_time), "start_time")

    # Each departure falls in exactly one band, b_k <= q_i < b_{k+1}
    # (eps makes the upper end strict, so a departure exactly at a boundary goes to the later band)
    tsp.addConstrs((quicksum(p[i, k] for k in range(K)) == 1 for i in range(n)), "one_band")
    tsp.addConstrs((q[i] >= bounds[k] - M * (1 - p[i, k]) for i in range(n) for k in range(K)), "band_lb")
    tsp.addConstrs((q[i] <= bounds[k + 1] - eps + M * (1 - p[i, k]) for i in range(n) for k in range(K)), "band_ub")

    # Time passes along the route with no waiting (also eliminates subtours)
    for i in range(n):
        for j in range(1, n):
            if i != j:
                tsp.addConstr(s[j] >= q[i] + tau[i, j] - M * (1 - x[i, j]), f"arrive_lb[{i},{j}]")
                tsp.addConstr(s[j] <= q[i] + tau[i, j] + M * (1 - x[i, j]), f"arrive_ub[{i},{j}]")

    # Travel time of the leg entering each node
    for i in range(n):
        for j in range(n):
            if i != j:
                tsp.addConstr(T[j] >= tau[i, j] - M * (1 - x[i, j]), f"leg_time[{i},{j}]")

    tsp.optimize()
    if tsp.Status != GRB.OPTIMAL:
        return None

    route = [0]
    while True:
        nxt = next(j for j in range(n) if j != route[-1] and x[route[-1], j].X > 0.5)
        route.append(nxt)
        if nxt == 0:
            break

    # Report the travel time of the chosen route by driving it exactly
    travel, _, _ = evaluate_route(route, travel_times, inspection_times, alpha, band_bounds, start_time)
    assert abs(travel - tsp.ObjVal) < 1e-3, f"MIP objective {tsp.ObjVal:.4f} != route travel {travel:.4f}"
    return route, travel

### Base Case: Free-Flow Travel Times

Setting every factor to 1 gives back the base TSP. This is the route the team would plan without congestion data.

In [ ]:
alpha_free_flow = np.ones((num_nodes, num_nodes, len(band_factors)))
base_route, base_travel = run_td_tsp(travel_time_matrix, inspection_times, alpha_free_flow,
                                     band_bounds, START_TIME, verbose=False)
_, base_return, _ = evaluate_route(base_route, travel_time_matrix, inspection_times,
                                   alpha_free_flow, band_bounds, START_TIME)

print(f"Base route: {to_labels(base_route)}")
print(f"Free-flow travel time: {base_travel:.2f} min, back at depot {to_clock(base_return)}")

### Results: TomTom Time-of-Day Factors

In [ ]:
alpha_tomtom = build_alpha(band_factors, num_nodes)
td_route, td_travel = run_td_tsp(travel_time_matrix, inspection_times, alpha_tomtom,
                                 band_bounds, START_TIME)
_, td_return, td_legs = evaluate_route(td_route, travel_time_matrix, inspection_times,
                                       alpha_tomtom, band_bounds, START_TIME)

# The base route driven under congestion
base_cong_travel, base_cong_return, _ = evaluate_route(base_route, travel_time_matrix, inspection_times,
                                                       alpha_tomtom, band_bounds, START_TIME)

print(f"Optimal route under congestion: {to_labels(td_route)}")
print(" -> ".join(short_names[i] for i in td_route))
print(f"Total travel time: {td_travel:.2f} min, back at depot {to_clock(td_return)}")
print(f"Base route under congestion: {base_cong_travel:.2f} min, back at depot {to_clock(base_cong_return)}")
print(f"Increase over free flow: {td_travel - base_travel:.2f} min (+{(td_travel / base_travel - 1) * 100:.1f}%)")

#### Schedule of the Optimal Route

In [ ]:
schedule_df = pd.DataFrame(td_legs)
schedule_df["leg"] = [f"{short_names[a]} -> {short_names[b]}" for a, b in zip(schedule_df["from"], schedule_df["to"])]
schedule_df["depart_clock"] = schedule_df["depart"].apply(to_clock)
schedule_df["band"] = [band_labels[k] for k in schedule_df["band"]]
schedule_df["free_flow_min"] = [travel_time_matrix[a, b] for a, b in zip(schedule_df["from"], schedule_df["to"])]
schedule_df = schedule_df[["leg", "depart_clock", "band", "factor", "free_flow_min", "travel_min"]].round(2)
schedule_df.to_csv(os.path.join(RESULTS_PATH, "extension_1_schedule.csv"), index=False)
schedule_df

### Check Against Brute Force

In [ ]:
ranked = brute_force_td_tsp(travel_time_matrix, inspection_times, alpha_tomtom, band_bounds, START_TIME)
print(f"Brute-force optimum: {to_labels(ranked[0][2])}, {ranked[0][0]:.2f} min")
assert abs(ranked[0][0] - td_travel) < 1e-6

print("\nTop 5 routes under congestion:")
for travel, ret, route in ranked[:5]:
    print(f"  {travel:6.2f} min  back {to_clock(ret)}  {to_labels(route)}")

#### Congestion Factor over the Day

The step line shows the factor for each 2-hour band. The dots mark when each leg of the optimal route starts.

In [ ]:
def plot_factor_profile(day_df, band_bounds, legs, file_path, start_hour=6, end_hour=20):
    keep = [k for k in range(len(band_bounds) - 1) if band_bounds[k] < end_hour * 60]
    hours = np.array([band_bounds[k] for k in keep] + [band_bounds[keep[-1] + 1]]) / 60
    factors = day_df["factor"].to_numpy()[keep]
    factors = np.append(factors, factors[-1])

    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.step(hours, factors, where="post", color="#2a78d6", linewidth=2)
    for leg in legs:
        ax.plot(leg["depart"] / 60, leg["factor"], "o", markersize=8, color="#2a78d6",
                markeredgecolor="white", markeredgewidth=2, zorder=3)
    ax.annotate("Leg start times of the optimal route", xy=(legs[0]["depart"] / 60, legs[0]["factor"]),
                xytext=(legs[0]["depart"] / 60 + 0.1, 1.75), color="#52514e", fontsize=9,
                arrowprops=dict(arrowstyle="-", color="#9a9893", lw=1))

    ax.axhline(1.0, color="#9a9893", linewidth=1, linestyle="--")
    ax.text(end_hour - 1.9, 1.02, "Free flow", color="#52514e", fontsize=9)
    ax.set_xlim(start_hour, end_hour)
    ax.set_xticks(range(start_hour, end_hour + 1, 2))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(start_hour, end_hour + 1, 2)])
    ax.set_ylim(0.9, 2.0)
    ax.set_xlabel("Time of day")
    ax.set_ylabel("Travel time relative to free flow")
    ax.set_title("Weekday congestion factor in Singapore (TomTom 2025)", loc="left")
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(file_path, dpi=200)
    plt.show()

plot_factor_profile(day_df, band_bounds, td_legs,
                    os.path.join(RESULTS_PATH, "extension_1_congestion_profile.png"))

### Sensitivity Analysis

We vary one input at a time and keep the others at their main values (TomTom factors, 8:30 start, 60 minutes per centre). For each setting we compare:

- **Re-optimised**: the best route for that setting.
- **Keep base route**: the free-flow route, driven at that setting.

The difference is the time saved by planning with congestion data.


#### 1. Directional Congestion

In each run, legs in one direction between the two clusters get an extra factor $\beta$ in one time band, with $\beta$ from 1.0 to 2.0.

In [ ]:
def run_beta_sweep(betas, beta_bands, beta_arcs, label):
    rows = []
    for beta in betas:
        alpha = build_alpha(band_factors, num_nodes, beta=beta, beta_bands=beta_bands, beta_arcs=beta_arcs)
        route, travel = run_td_tsp(travel_time_matrix, inspection_times, alpha,
                                   band_bounds, START_TIME, verbose=False)
        keep_travel, _, _ = evaluate_route(base_route, travel_time_matrix, inspection_times,
                                           alpha, band_bounds, START_TIME)
        rows.append({"case": label, "beta": beta, "route": to_labels(route),
                     "reoptimised_travel": travel, "base_route_travel": keep_travel,
                     "time_saved": keep_travel - travel, "route_changed": route != base_route})
    return pd.DataFrame(rows)

BETAS = np.round(np.arange(1.0, 2.0001, 0.02), 2)
band_index = {label: k for k, label in enumerate(band_labels)}

**1a. City-bound legs in the morning peak.** This follows the ERP pattern: the AYE towards the city is priced in the morning.

In [ ]:
sweep_am = run_beta_sweep(BETAS, [band_index["08:00-10:00"]], CITY_BOUND_ARCS, "City-bound, 08:00-10:00")
print(f"Route changes for any beta in [1, 2]: {sweep_am['route_changed'].any()}")
sweep_am[["beta", "reoptimised_travel", "base_route_travel", "time_saved"]].iloc[::10].round(2)

**1b. Congestion when the route crosses between clusters.** The optimal route crosses to the east cluster late in the morning and back to the west mid-afternoon (see the schedule above). This is a stress test: we found no data showing extra congestion in these bands.

In [ ]:
sweep_east = run_beta_sweep(BETAS, [band_index["10:00-12:00"]], CITY_BOUND_ARCS, "City-bound, 10:00-12:00")
sweep_west = run_beta_sweep(BETAS, [band_index["14:00-16:00"]], WEST_BOUND_ARCS, "West-bound, 14:00-16:00")

for df in [sweep_east, sweep_west]:
    changed = df[df["route_changed"]]
    if len(changed) > 0:
        first = changed.iloc[0]
        print(f"{first['case']}: route changes from beta = {first['beta']:.2f} to {first['route']}")
    print(f"  at beta = 2.0: {df.iloc[-1]['time_saved']:.2f} min saved by re-optimising")

sweeps_df = pd.concat([sweep_am, sweep_east, sweep_west], ignore_index=True)
sweeps_df.round(3).to_csv(os.path.join(RESULTS_PATH, "extension_1_directional_sweep.csv"), index=False)

In [ ]:
def plot_beta_sweep(df, file_path):
    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.plot(df["beta"], df["base_route_travel"], color="#eb6834", linewidth=2, label="Keep base route")
    ax.plot(df["beta"], df["reoptimised_travel"], color="#2a78d6", linewidth=2, label="Re-optimised")
    last = df.iloc[-1]
    ax.text(last["beta"] + 0.02, last["base_route_travel"], "Keep base route", color="#0b0b0b", va="center", fontsize=9)
    ax.text(last["beta"] + 0.02, last["reoptimised_travel"], "Re-optimised", color="#0b0b0b", va="center", fontsize=9)
    ax.set_xlim(1.0, 2.25)
    ax.set_xticks(np.round(np.arange(1.0, 2.01, 0.2), 1))
    ax.set_xlabel("Extra congestion weight on city-bound legs, 10:00-12:00 (beta)")
    ax.set_ylabel("Total travel time (min)")
    ax.set_title("Planning with congestion data saves more as late-morning city-bound congestion grows", loc="left")
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False, loc="upper left")
    fig.tight_layout()
    fig.savefig(file_path, dpi=200)
    plt.show()

plot_beta_sweep(sweep_east, os.path.join(RESULTS_PATH, "extension_1_directional_sweep.png"))

Parts 2 and 3 use the TomTom factors for every leg. The function below runs one input at a time.

In [ ]:
def run_sensitivity(parameter, values):
    rows = []
    for value in values:
        start = value if parameter == "start_time" else START_TIME
        insp = make_inspection_times(value if parameter == "inspection_min" else INSPECTION_TIME_PER_CENTRE)
        route, travel = run_td_tsp(travel_time_matrix, insp, alpha_tomtom, band_bounds, start, verbose=False)
        _, ret, _ = evaluate_route(route, travel_time_matrix, insp, alpha_tomtom, band_bounds, start)
        keep_travel, _, _ = evaluate_route(base_route, travel_time_matrix, insp, alpha_tomtom, band_bounds, start)
        rows.append({"parameter": parameter, "value": value,
                     "start": to_clock(start), "inspection_min": insp[1],
                     "route": to_labels(route), "reoptimised_travel": travel, "base_route_travel": keep_travel,
                     "time_saved": keep_travel - travel, "back_at_depot": to_clock(ret),
                     "route_changed": route != base_route})
    return pd.DataFrame(rows)

START_TIMES = [7 * 60, 7 * 60 + 30, 8 * 60, 8 * 60 + 30, 9 * 60, 9 * 60 + 30, 10 * 60]
INSPECTION_MINUTES = [30, 45, 60, 75, 90]

sens_start = run_sensitivity("start_time", START_TIMES)
sens_inspection = run_sensitivity("inspection_min", INSPECTION_MINUTES)

sensitivity_df = pd.concat([sens_start, sens_inspection], ignore_index=True)
sensitivity_df.round(3).to_csv(os.path.join(RESULTS_PATH, "extension_1_sensitivity.csv"), index=False)

#### 2. Start Time

In [ ]:
sens_start[["start", "route", "reoptimised_travel", "base_route_travel", "time_saved", "back_at_depot"]].round(2)

#### 3. Inspection Time per Centre

In [ ]:
sens_inspection[["inspection_min", "route", "reoptimised_travel", "base_route_travel", "time_saved", "back_at_depot"]].round(2)

In [ ]:
def plot_sensitivity(sens_start, sens_inspection, file_path):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
    panels = [
        (axes[0], sens_start, [v / 60 for v in sens_start["value"]], "Start time",
         lambda ax: (ax.set_xticks([v / 60 for v in START_TIMES]),
                     ax.set_xticklabels(list(sens_start["start"])))),
        (axes[1], sens_inspection, list(sens_inspection["value"]), "Inspection time per centre (min)",
         lambda ax: ax.set_xticks(INSPECTION_MINUTES)),
    ]
    for ax, df, xs, xlabel, set_ticks in panels:
        ax.plot(xs, df["base_route_travel"], color="#eb6834", linewidth=2,
                marker="o", markersize=8, markeredgecolor="white", markeredgewidth=2, label="Keep base route")
        ax.plot(xs, df["reoptimised_travel"], color="#2a78d6", linewidth=2,
                marker="o", markersize=8, markeredgecolor="white", markeredgewidth=2, label="Re-optimised")
        set_ticks(ax)
        ax.set_xlabel(xlabel)
        for side in ["top", "right"]:
            ax.spines[side].set_visible(False)
        ax.grid(axis="y", alpha=0.25)
    axes[0].set_ylabel("Total travel time (min)")
    axes[0].legend(frameon=False, loc="upper left")
    fig.suptitle("Re-planning for a different start or inspection time saves at most a couple of minutes",
                 x=0.01, ha="left", fontsize=12)
    fig.tight_layout()
    fig.savefig(file_path, dpi=200)
    plt.show()

plot_sensitivity(sens_start, sens_inspection, os.path.join(RESULTS_PATH, "extension_1_sensitivity.png"))

### Save Summary

In [ ]:
summary_df = pd.DataFrame([
    {"case": "Base (free flow)", "route": to_labels(base_route), "total_travel_min": base_travel,
     "back_at_depot": to_clock(base_return)},
    {"case": "Base route under TomTom congestion", "route": to_labels(base_route), "total_travel_min": base_cong_travel,
     "back_at_depot": to_clock(base_cong_return)},
    {"case": "Re-optimised under TomTom congestion", "route": to_labels(td_route), "total_travel_min": td_travel,
     "back_at_depot": to_clock(td_return)},
])
summary_df.round(2).to_csv(os.path.join(RESULTS_PATH, "extension_1_summary.csv"), index=False)
summary_df.round(2)